# Prefix Sum — Google L4 DSA Notes

## Solved Problems

1. [LC 560 — Subarray Sum Equals K](https://leetcode.com/problems/subarray-sum-equals-k/)
2. [LC 974 — Subarray Sums Divisible by K](https://leetcode.com/problems/subarray-sums-divisible-by-k/)
3. [LC 525 — Contiguous Array](https://leetcode.com/problems/contiguous-array/)
4. [LC 523 — Continuous Subarray Sum](https://leetcode.com/problems/continuous-subarray-sum/)
5. [LC 1590 — Make Sum Divisible by P](https://leetcode.com/problems/make-sum-divisible-by-p/)

---

# Prefix Sum Recognition

Ask:

> **Can I express the condition on a subarray using two prefix states?**

For a subarray from `i+1` to `j`:

`subarray_sum = prefix[j] - prefix[i]`

The important part is deciding **what prefix state to store**.

| Goal | Store | Reason |
|---|---|---|
| COUNT subarrays | `frequency[state]` | Every matching previous state creates a valid subarray |
| LONGEST subarray | `earliest[state]` | Earliest occurrence gives maximum length |
| SHORTEST subarray | `latest[state]` | Latest occurrence gives minimum length |
| Divisibility | `prefix % k` | Same remainder → difference divisible by `k` |
| Equal counts | `balance` | Same balance → equal counts |


# 1. LC 560 — Subarray Sum Equals K

**Pattern:** Prefix Sum + HashMap

### Recognition

The problem asks for the **number of subarrays** whose sum equals `k`.

That immediately suggests:

> **COUNT + subarray sum → Prefix Sum + frequency map**

### Derivation

Suppose the current prefix sum is:

`current_prefix`

We need an earlier prefix such that:

`current_prefix - previous_prefix = k`

Therefore:

`previous_prefix = current_prefix - k`

So at every position:

1. Calculate current prefix sum.
2. Check how many times `current_prefix - k` has appeared.
3. Add that frequency to the answer.
4. Store the current prefix sum.

### Important initialization

`prefix_freq = {0: 1}`

This represents the empty prefix before the array begins.


In [ ]:
from typing import List

class Solution:
    def subarraySum(self, nums: List[int], k: int) -> int:
        prefix_freq = {0: 1}
        prefix = 0
        ans = 0

        for num in nums:
            prefix += num

            needed = prefix - k
            ans += prefix_freq.get(needed, 0)

            prefix_freq[prefix] = prefix_freq.get(prefix, 0) + 1

        return ans


### Complexity

- Time: **O(n)**
- Space: **O(n)**

### Key Lesson

**COUNT → frequency of prefix states**

Do not store only whether a prefix exists. The same prefix can occur multiple times, and every occurrence can create a different valid subarray.


# 2. LC 974 — Subarray Sums Divisible by K

**Pattern:** Prefix Sum + Remainder + Frequency

### Recognition

The problem asks for subarrays whose sum is **divisible by `k`**.

Instead of storing the complete prefix sum, store:

`prefix % k`

### Derivation

For a subarray:

`subarray_sum = prefix[j] - prefix[i]`

We need:

`(prefix[j] - prefix[i]) % k == 0`

Therefore:

`prefix[j] % k == prefix[i] % k`

So two equal prefix remainders form a valid subarray.

### Recognition Rule

> **Divisibility → prefix remainder**

And because the question asks for a **COUNT**:

> **COUNT → frequency**


In [ ]:
from typing import List

class Solution:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        remainder_freq = {0: 1}
        prefix = 0
        ans = 0

        for num in nums:
            prefix += num
            remainder = prefix % k

            ans += remainder_freq.get(remainder, 0)
            remainder_freq[remainder] = remainder_freq.get(remainder, 0) + 1

        return ans


### Complexity

- Time: **O(n)**
- Space: **O(k)** in terms of distinct possible remainders.

### Key Lesson

If two prefix sums leave the **same remainder** when divided by `k`, their difference is divisible by `k`.


# 3. LC 525 — Contiguous Array

**Pattern:** Prefix Sum + Balance

### Recognition

We need the longest subarray containing equal numbers of `0` and `1`.

Turn the problem into a prefix-state problem:

- `1 → +1`
- `0 → -1`

Now ask:

> When does a subarray have equal numbers of 0s and 1s?

When its total balance is `0`.

If the same balance occurs at two positions, the portion between them has net balance `0`.

Therefore:

> **Same balance → equal counts**

### Why earliest index?

We want the **LONGEST** subarray.

For a repeated balance, using the earliest occurrence gives the largest possible distance.

So:

> **LONGEST → earliest index**


In [ ]:
from typing import List

class Solution:
    def findMaxLength(self, nums: List[int]) -> int:
        first_index = {0: -1}
        balance = 0
        ans = 0

        for i, num in enumerate(nums):
            if num == 1:
                balance += 1
            else:
                balance -= 1

            if balance in first_index:
                ans = max(ans, i - first_index[balance])
            else:
                first_index[balance] = i

        return ans


### Complexity

- Time: **O(n)**
- Space: **O(n)**

### Key Lesson

For equal-count problems, create a **balance**.

`0 → -1`, `1 → +1`

Then:

> **Same balance at two indices → equal counts between them.**


# 4. LC 523 — Continuous Subarray Sum

**Pattern:** Prefix Sum + Remainder

### Recognition

We need a subarray whose sum is a multiple of `k`.

Again:

`subarray_sum = prefix[j] - prefix[i]`

For divisibility:

`(prefix[j] - prefix[i]) % k == 0`

Therefore:

`prefix[j] % k == prefix[i] % k`

So we store the first index at which each remainder appeared.

### Why earliest index?

The problem only needs a valid subarray of length at least 2.

Keeping the earliest index maximizes the distance to a later matching remainder.

### Important condition

When the same remainder is seen again:

`i - first_index[remainder] >= 2`


In [ ]:
from typing import List

class Solution:
    def checkSubarraySum(self, nums: List[int], k: int) -> bool:
        first_index = {0: -1}
        prefix = 0

        for i, num in enumerate(nums):
            prefix += num
            remainder = prefix % k

            if remainder in first_index:
                if i - first_index[remainder] >= 2:
                    return True
            else:
                first_index[remainder] = i

        return False


### Complexity

- Time: **O(n)**
- Space: **O(k)**

### Key Lesson

For divisibility problems:

`same remainder → divisible difference`

For a longest/valid subarray, keep the **earliest** index.


# 5. LC 1590 — Make Sum Divisible by P

**Pattern:** Prefix Sum + Remainder + Latest Index

### Recognition

We want to remove the **shortest** subarray so that the remaining sum is divisible by `p`.

First calculate:

`total_remainder = total_sum % p`

The removed subarray must have:

`removed_sum % p = total_remainder`

For a current prefix remainder `current_rem`, we need an earlier prefix remainder satisfying:

`(current_rem - previous_rem) % p = total_remainder`

Therefore:

`previous_rem = (current_rem - total_remainder) % p`

### Why latest index?

We need the **SHORTEST** subarray.

For the same required prefix state, the latest previous index gives the shortest distance.

So:

> **SHORTEST → latest index**


In [ ]:
from typing import List

class Solution:
    def minSubarray(self, nums: List[int], p: int) -> int:
        total_remainder = sum(nums) % p

        if total_remainder == 0:
            return 0

        latest_index = {0: -1}
        prefix = 0
        ans = len(nums)

        for i, num in enumerate(nums):
            prefix = (prefix + num) % p

            current_rem = prefix
            needed = (current_rem - total_remainder) % p

            if needed in latest_index:
                ans = min(ans, i - latest_index[needed])

            latest_index[current_rem] = i

        return ans if ans < len(nums) else -1


### Complexity

- Time: **O(n)**
- Space: **O(n)**

### Key Lesson

This problem is the perfect example of:

> **SHORTEST → latest index**

The prefix state is a **remainder**, not the raw prefix sum.


# Prefix Sum Master Cheat Sheet

### 1. COUNT

Store:

```python
frequency[state]
```

Example: **LC 560, LC 974**

---

### 2. LONGEST

Store:

```python
earliest[state]
```

Example: **LC 525, LC 523**

---

### 3. SHORTEST

Store:

```python
latest[state]
```

Example: **LC 1590**

---

### 4. DIVISIBILITY

Replace the prefix sum with:

```python
prefix % k
```

Because:

```text
same remainder
      ↓
difference is divisible by k
```

---

### 5. EQUAL COUNTS

Create a balance:

```text
category A → +1
category B → -1
```

Same balance means equal counts.

---

## The Recognition Question

When you see a new problem, ask:

> **Can I describe the desired subarray using the relationship between two prefix states?**

If yes, identify:

1. **What is the state?** — sum / remainder / balance / etc.
2. **What is the goal?** — count / longest / shortest / existence?
3. **What should I store?** — frequency / earliest / latest index?
